1. Setup

In [1]:
# =========================================================
# Librerías base
# =========================================================
import os
import sys
import importlib
import time
import joblib

# Manejo de datos
import pandas as pd
import numpy as np

# Métricas para clasificación binaria
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Pesos de clase para desbalance
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------------------------------
# Detecta la raíz del proyecto
# ---------------------------------------------------------
PROJECT_ROOT = os.path.abspath("..")

# ---------------------------------------------------------
# Agrega la raíz al path para importar módulos propios
# ---------------------------------------------------------
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# ---------------------------------------------------------
# Importa configuración del proyecto
# ---------------------------------------------------------
import utils.config as config
importlib.reload(config)

# ---------------------------------------------------------
# Importa servicio de entrenamiento reutilizable
# ---------------------------------------------------------
from services.training.training_service import TrainingService

print("FULL_ARTIFACTS_DIR:", config.FULL_ARTIFACTS_DIR)

FULL_ARTIFACTS_DIR: /home/bryan_santanderaragon/Proyecto_MBD/data/artifacts/full


2. Cargar los artifacts binarios

In [2]:
# ---------------------------------------------------------
# Carga los datasets binarios generados en
# 02_preprocessing_full.ipynb
# ---------------------------------------------------------
X_train = pd.read_parquet(
    os.path.join(config.FULL_ARTIFACTS_DIR, "X_train_full_binary.parquet")
)

X_test = pd.read_parquet(
    os.path.join(config.FULL_ARTIFACTS_DIR, "X_test_full_binary.parquet")
)

y_train = pd.read_csv(
    os.path.join(config.FULL_ARTIFACTS_DIR, "y_train_full_binary.csv")
).squeeze("columns")

y_test = pd.read_csv(
    os.path.join(config.FULL_ARTIFACTS_DIR, "y_test_full_binary.csv")
).squeeze("columns")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (17871216, 22)
X_test: (4467805, 22)
y_train: (17871216,)
y_test: (4467805,)


3. Revisar la distribución de las clases

In [3]:
# ---------------------------------------------------------
# Muestra la distribución de la variable objetivo binaria
# para ver el desbalance entre normal y ataque
# ---------------------------------------------------------
class_distribution = pd.Series(y_train).value_counts().sort_index()

print("Distribución de clases en y_train:")
print(class_distribution)


Distribución de clases en y_train:
label
0      637104
1    17234112
Name: count, dtype: int64


4. Calcular el peso de las clases

In [4]:
# ---------------------------------------------------------
# Calcula pesos para compensar el desbalance.
# En binario suele ser especialmente útil.
# ---------------------------------------------------------
classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_dict = dict(zip(classes, class_weights_array))



print("Pesos de clase:")
print(class_weights_dict)

# ---------------------------------------------------------
# Calcular peso de la clase positiva para XGBoost binario
# scale_pos_weight = negativos / positivos
# ---------------------------------------------------------
num_negative = class_distribution.loc[0]
num_positive = class_distribution.loc[1]

scale_pos_weight = num_negative / num_positive

print("scale_pos_weight:", scale_pos_weight)

Pesos de clase:
{np.int64(0): np.float64(14.025352218790024), np.int64(1): np.float64(0.5184838069985851)}
scale_pos_weight: 0.036967613997170266


Proceso de limpieza ara evitar sobrecargar la ram

In [5]:
# ---------------------------------------------------------
# LIMPIEZA INTERMEDIA 1
# Mantiene solo lo necesario para entrenar
# ---------------------------------------------------------
import gc
import psutil
import os

vars_to_delete = [
    "class_distribution"
]

for var_name in vars_to_delete:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

process = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / 1024 / 1024
print(f"RAM usada después de limpieza intermedia 1: {memory_mb:.2f} MB")

RAM usada después de limpieza intermedia 1: 6128.04 MB


5. Crear la instancia para el training

In [6]:
# ---------------------------------------------------------
# Instancia del servicio de entrenamiento
# ---------------------------------------------------------
trainer = TrainingService()

# ---------------------------------------------------------
# Número de clases del problema binario
# ---------------------------------------------------------
num_classes = len(np.unique(y_train))
print("Número de clases:", num_classes)

Número de clases: 2


6. Entrenar Random Forest Binario

In [7]:
# ---------------------------------------------------------
# Entrenamiento de Random Forest para clasificación binaria
# usando class_weight='balanced'
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

rf_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=config.RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_elapsed = time.time() - rf_start
print(f"✔ Random Forest binario entrenado en {rf_elapsed:.2f} seg")

✔ Random Forest binario entrenado en 1600.46 seg


7. Entrenar XGBoost binario

In [8]:
# ---------------------------------------------------------
# Entrenamiento de XGBoost para clasificación binaria.
# El TrainingService debería detectar num_classes=2
# y usar objective='binary:logistic'
# ---------------------------------------------------------
from xgboost import XGBClassifier
xgb_start = time.time()

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=config.RANDOM_STATE,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_elapsed = time.time() - xgb_start
print(f"✔ XGBoost binario balanceado entrenado en {xgb_elapsed:.2f} seg")

✔ XGBoost binario balanceado entrenado en 226.60 seg


8. Entrenar LightGBM binario

In [9]:
# ---------------------------------------------------------
# Entrenamiento de LightGBM para clasificación binaria
# ---------------------------------------------------------
from lightgbm import LGBMClassifier

lgb_start = time.time()

lgb_model = LGBMClassifier(
    objective="binary",
    n_estimators=100,
    learning_rate=0.1,
    random_state=config.RANDOM_STATE,
    class_weight="balanced"
)

lgb_model.fit(X_train, y_train)

lgb_elapsed = time.time() - lgb_start
print(f"✔ LightGBM binario balanceado entrenado en {lgb_elapsed:.2f} seg")

[LightGBM] [Info] Number of positive: 17234112, number of negative: 637104
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.097202 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1927
[LightGBM] [Info] Number of data points in the train set: 17871216, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
✔ LightGBM binario balanceado entrenado en 322.63 seg


9. Función de evaluación binaria

In [10]:
# ---------------------------------------------------------
# Función para evaluar modelos binarios
# Calcula:
# - accuracy
# - precision
# - recall
# - f1
# - roc_auc (si el modelo entrega probabilidades)
# ---------------------------------------------------------
def evaluate_binary_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="binary",
        zero_division=0
    )

    # -----------------------------------------------------
    # Intenta calcular ROC-AUC usando probabilidades.
    # Si el modelo no tiene predict_proba, se deja en NaN.
    # -----------------------------------------------------
    try:
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    except Exception:
        y_prob = None
        roc_auc = np.nan

    results = {
        "model": model_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

    print(f"\n==================== {model_name} ====================")
    print(classification_report(y_test, y_pred, zero_division=0))

    return results, y_pred, y_prob

10. Evaluar Random Forest Binario

In [11]:
rf_results, rf_pred, rf_prob = evaluate_binary_model(
    model=rf_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Random Forest Binary Full"
)


==================== Random Forest Binary Full ====================
              precision    recall  f1-score   support

           0       0.64      0.99      0.78    159276
           1       1.00      0.98      0.99   4308529

    accuracy                           0.98   4467805
   macro avg       0.82      0.99      0.89   4467805
weighted avg       0.99      0.98      0.98   4467805



11. Evaluar XGBoost Binario

In [12]:
xgb_results, xgb_pred, xgb_prob = evaluate_binary_model(
    model=xgb_model,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost Binary Full"
)


==================== XGBoost Binary Full ====================
              precision    recall  f1-score   support

           0       0.62      0.99      0.76    159276
           1       1.00      0.98      0.99   4308529

    accuracy                           0.98   4467805
   macro avg       0.81      0.99      0.88   4467805
weighted avg       0.99      0.98      0.98   4467805



12. Evaluar LightGBM binario

In [13]:
lgb_results, lgb_pred, lgb_prob = evaluate_binary_model(
    model=lgb_model,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM Binary Full"
)


==================== LightGBM Binary Full ====================
              precision    recall  f1-score   support

           0       0.62      0.99      0.77    159276
           1       1.00      0.98      0.99   4308529

    accuracy                           0.98   4467805
   macro avg       0.81      0.99      0.88   4467805
weighted avg       0.99      0.98      0.98   4467805



13. Comparación de resultados binarios

In [14]:
# ---------------------------------------------------------
# Consolida resultados en una tabla
# ---------------------------------------------------------
results_df = pd.DataFrame([rf_results, xgb_results, lgb_results])

# ---------------------------------------------------------
# Ordenar por F1 de mayor a menor
# ---------------------------------------------------------
results_df = results_df.sort_values(by="f1", ascending=False).reset_index(drop=True)

results_df

,model,accuracy,precision,recall,f1,roc_auc
0,Random Forest Binary Full,0.980129,0.999739,0.979650,0.989592,0.998654
1,LightGBM Binary Full,0.978305,0.999759,0.977739,0.988626,0.998965
2,XGBoost Binary Full,0.977875,0.999725,0.977326,0.988399,0.998773


14. Guardar resultados binarios

In [15]:
# ---------------------------------------------------------
# Guarda la tabla comparativa del flujo binario full
# ---------------------------------------------------------
results_path = os.path.join(config.FULL_ARTIFACTS_DIR, "binary_model_results_full.csv")
results_df.to_csv(results_path, index=False)

print("Resultados binarios full guardados en:")
print(results_path)

Resultados binarios full guardados en:
/home/bryan_santanderaragon/Proyecto_MBD/data/artifacts/full/binary_model_results_full.csv


15. Guardar modelos binarios

In [16]:
# ---------------------------------------------------------
# Guarda los modelos binarios entrenados
# ---------------------------------------------------------
joblib.dump(
    rf_model,
    os.path.join(config.FULL_ARTIFACTS_DIR, "rf_binary_full.pkl")
)

joblib.dump(
    xgb_model,
    os.path.join(config.FULL_ARTIFACTS_DIR, "xgb_binary_full.pkl")
)

joblib.dump(
    lgb_model,
    os.path.join(config.FULL_ARTIFACTS_DIR, "lgb_binary_full.pkl")
)

print("Modelos binarios full guardados correctamente.")

Modelos binarios full guardados correctamente.


16. Guardar predicciones binarias

In [17]:
# ---------------------------------------------------------
# Guarda predicciones para análisis posterior
# ---------------------------------------------------------
pd.DataFrame({"y_true": y_test, "y_pred": rf_pred}).to_csv(
    os.path.join(config.FULL_ARTIFACTS_DIR, "rf_binary_predictions_full.csv"),
    index=False
)

pd.DataFrame({"y_true": y_test, "y_pred": xgb_pred}).to_csv(
    os.path.join(config.FULL_ARTIFACTS_DIR, "xgb_binary_predictions_full.csv"),
    index=False
)

pd.DataFrame({"y_true": y_test, "y_pred": lgb_pred}).to_csv(
    os.path.join(config.FULL_ARTIFACTS_DIR, "lgb_binary_predictions_full.csv"),
    index=False
)

print("Predicciones binarias guardadas.")

Predicciones binarias guardadas.


18. Identificar el mejor modelo binario

In [18]:
# ---------------------------------------------------------
# Identifica el mejor modelo según F1
# ---------------------------------------------------------
best_model = results_df.iloc[0]

print("Mejor modelo binario del flujo full:")
print(best_model)

Mejor modelo binario del flujo full:
model        Random Forest Binary Full
accuracy                      0.980129
precision                     0.999739
recall                         0.97965
f1                            0.989592
roc_auc                       0.998654
Name: 0, dtype: object


Limpiar memoria

In [19]:
# ---------------------------------------------------------
# Liberar variables pesadas de memoria
# ---------------------------------------------------------
import gc

vars_to_delete = [
    "df", "df_clean", "df_encoded", "df_full", "df_full_sample",
    "X", "X_train", "X_test", "X_train_scaled", "X_test_scaled",
    "y", "y_train", "y_test", "y_train_enc", "y_test_enc",
    "y_pred", "y_pred_eval", "y_pred_prob", "y_pred_prob_bin",
    "rf_model", "xgb_model", "lgb_model", "dnn_model",
    "sample_weight", "class_distribution", "results_df",
    "multiclass_results", "binary_results", "multiclass_comparison", "binary_comparison"
]

for var_name in vars_to_delete:
    if var_name in globals():
        del globals()[var_name]

# ---------------------------------------------------------
# Forzar garbage collector
# ---------------------------------------------------------
collected = gc.collect()
print(f"Objetos liberados por gc: {collected}")
print("Limpieza de memoria completada.")

Objetos liberados por gc: 90
Limpieza de memoria completada.
